# Build Graph using Cognee (Neo4j via Docker configuration)

In [1]:
# Test connection to Neo4j db
from cognee.infrastructure.databases.graph import get_graph_engine

graph = await get_graph_engine()
print(await graph.query("RETURN 1 AS ok"))


2026-11-13T21:28:54.992438 [info     ] Deleted old log file: C:\Users\moote\.cognee\logs\2026-11-12_17-07-25.log [cognee.shared.logging_utils]

2026-11-13T21:28:54.994448 [info     ] Log file created at: C:\Users\moote\.cognee\logs\2026-11-13_21-28-54.log [cognee.shared.logging_utils] log_file=C:\Users\moote\.cognee\logs\2026-11-13_21-28-54.log

2026-11-13T21:28:54.996448 [warning  ] Cognee 1.0 changes: New API — remember/recall/forget/improve (V1 add/cognify/search still work). Session memory enabled by default (CACHING=false to disable). Multi-user access control on by default (ENABLE_BACKEND_ACCESS_CONTROL=false to disable). Agents (@cognee.agent) auto-verified on registration. See https://docs.cognee.ai/ [cognee.shared.logging_utils]

2026-11-13T21:28:54.998448 [info     ] Logging initialized            [cognee.shared.logging_utils] cognee_version=1.6.7 database_path=D:\cognee_neo4j\system\databases os_info='Windows 10 (10.0.26300)' python_version=3.11.6 structlog_version=25.5.0



[{'ok': 1}]


In [2]:
from pathlib import Path
from time import perf_counter

import cognee

# Delete existing data
# await cognee.prune.prune_data()
# await cognee.prune.prune_system(metadata=True)
# await cognee.forget(everything=True)

# Data
data_path = "../../data/silver"

files = sorted(
    path for path in Path(data_path).rglob("*.md")
    if path.is_file() and path.stat().st_size > 0
)
for path in files:
    print(f"  - {path.relative_to(data_path)}")

# Ingestion
start = perf_counter()
await cognee.remember(
[str(path) for path in files],
dataset_name="docspace",
self_improvement=False,
)
print(f"Ingestion completed {(perf_counter() - start) / 60:.1f} min")

  - bitcoin\A Peer-to-Peer Electronic Cash System.md
  - bitcoin\Fact Sheet iShares.md
  - cyber\Exploring Cryptocurrency.md
  - defi\BIS bulletin-76.md
  - reports\21Shares Report June 2025.md
  - reports\crypto_global\weekly\CryptoGlobal_Weekly_2026-07-11.md
  - reports\crypto_global\weekly\CryptoGlobal_Weekly_2026-07-13.md
  - reports\imf\Crypto Assets Monitor.md
  - reports\imf\Crypto Monitor Q1 2026.md
  - reports\imf\GMM Special Feature - Crypto Monitor October 2025.md
  - reports\imf\Workshop on Fraud and Investigations Involving Crypto Assets.md
  - stablecoins\Brief remarks on stablecoins.md
  - stablecoins\bulletin-108-stablecoin-growth-policy-challenges-and-approaches.md
  - stablecoins\bulletin-125-stablecoin-remuneration-centralised-exchanges.md



setup plugin alembic.autogenerate.schemas

setup plugin alembic.autogenerate.tables

setup plugin alembic.autogenerate.types

setup plugin alembic.autogenerate.constraints

setup plugin alembic.autogenerate.defaults

setup plugin alembic.autogenerate.comments

setup plugin alembic.ext.checkconstraint_byname

Using database: sqlite+aiosqlite:///D:\cognee_neo4j\system\databases/cognee_db

Context impl SQLiteImpl.      

Will assume non-transactional DDL.

Relational migrations applied (target head).

Applying migration 'namespace_entity_type_node_ids' (cognee 1.2.0).

2026-11-13T21:29:19.577403 [info     ] Retrieved 0 nodes and 0 edges in 0.53 seconds [Neo4jAdapter]

Applying migration 'namespace_edge_type_point_ids' (cognee 1.2.0).

2026-11-13T21:29:19.642984 [info     ] Retrieved 0 nodes and 0 edges in 0.02 seconds [Neo4jAdapter]

EdgeType point id migration done (0 edge text(s) considered).

Applying migration 'postgres_graph_provenance_columns' (cognee 1.2.2).

Applying migration 'l

Ingestion completed 3.7 min


# Ingestion Checks

In [3]:
from collections import Counter, defaultdict

from cognee.context_global_variables import set_database_global_context_variables
from cognee.infrastructure.databases.graph import get_graph_engine

# 1. Get ingested documents
datasets = await cognee.datasets.list_datasets()
dataset = next((d for d in datasets if d.name == "docspace"), None)

if dataset is None:
    raise ValueError("Dataset not found: docspace")

documents = await cognee.datasets.list_data(dataset.id)

# 2. Get graph data
async with set_database_global_context_variables(
    dataset.id, dataset.owner_id
):
    graph = await get_graph_engine()
    nodes, edges = await graph.get_graph_data()

node_by_id = {
    node_id: props
    for node_id, props in nodes
}

types = Counter(
    props.get("type", "Unknown")
    for _, props in nodes
)

graph_docs = {
    node_id: props
    for node_id, props in node_by_id.items()
    if props.get("type") in ("TextDocument", "Document")
}

chunks = {
    node_id: props
    for node_id, props in node_by_id.items()
    if props.get("type") == "DocumentChunk"
}

chunks_by_doc = defaultdict(set)
parents_by_chunk = defaultdict(set)

for source, target, relation, props in edges:
    source, target = source, target

    if source in graph_docs and target in chunks:
        chunks_by_doc[source].add(target)
        parents_by_chunk[target].add(source)

    elif target in graph_docs and source in chunks:
        chunks_by_doc[target].add(source)
        parents_by_chunk[source].add(target)

# Detect chunks without source document
orphan_chunks = [
    cid for cid in chunks
    if not parents_by_chunk[cid]
]

# Detect chunks with multiple source documents
multi_parent_chunks = [
    cid for cid, parents in parents_by_chunk.items()
    if len(parents) > 1
]

# 3. Summary
print("=" * 70)
print("Summary for dataset: docspace")
print("=" * 70)

print(f"Local files       : {len(files)}")
print(f"Ingested documents : {len(documents)}")
print(f"Graph documents   : {len(graph_docs)}")
print(f"Graph chunks      : {len(chunks)}")
print(f"Orphan chunks    : {len(orphan_chunks)}")
print(f"Multi-parent chunks: {len(multi_parent_chunks)}")
print(f"Total nodes          : {len(nodes)}")
print(f"Total edges     : {len(edges)}")

print("\nINGESTED DOCUMENTS")
for doc in documents:
    print(f"- {doc.name} | {doc.id}")

print("\nCHUNKS PER GRAPH DOCUMENT")
for doc_id, props in sorted(
    graph_docs.items(),
    key=lambda x: str(x[1].get("name", ""))
):
    name = props.get("name", doc_id)
    count = len(chunks_by_doc[doc_id])
    print(f"- {name}: {count} chunks")

print("\nNODE TYPES")
for node_type, count in types.most_common():
    print(f"- {node_type}: {count}")

print("\nCHECKS")
if files and len(files) != len(documents):
    print("[WARNING] Number of local files differs from dataset")

if len(documents) != len(graph_docs):
    print("[INFO] Number of documents differs between dataset and graph")

if orphan_chunks:
    print(f"[WARNING] {len(orphan_chunks)} chunks without direct link to a document")

if (
    (not files or len(files) == len(documents))
    and len(documents) == len(graph_docs)
    and not orphan_chunks
):
    print("[OK] No discrepancies detected")


2026-11-13T21:51:42.472399 [info     ] Retrieved 416 nodes and 1013 edges in 0.59 seconds [Neo4jAdapter]


Summary for dataset: docspace
Local files       : 14
Ingested documents : 14
Graph documents   : 14
Graph chunks      : 47
Orphan chunks    : 0
Multi-parent chunks: 0
Total nodes          : 416
Total edges     : 1013

INGESTED DOCUMENTS
- BIS bulletin-76 | d4b0d1e9-c043-4273-8a10-a81cec510df9
- Crypto Monitor Q1 2026 | 500396b1-dcaf-4fdd-a672-cd4c448606c4
- bulletin-108-stablecoin-growth-policy-challenges-and-approaches | 09553315-e6cc-428f-bb7c-b2556925562b
- CryptoGlobal_Weekly_2026-07-11 | ff51e0e7-0f73-4ac2-b228-108c5749d651
- A Peer-to-Peer Electronic Cash System | 491e7dd4-1027-4882-a2b1-248f8372177b
- Workshop on Fraud and Investigations Involving Crypto Assets | f5746feb-5ec4-4a3f-944f-fdb18691fe90
- Fact Sheet iShares | 4a66abd0-3baf-485e-a497-34059a8aa122
- 21Shares Report June 2025 | 555dcf06-0623-4daa-b9ce-306efec897ba
- Crypto Assets Monitor | 58b92a48-781d-4856-a801-ac94342845f4
- Brief remarks on stablecoins | ba941b9c-8895-4627-af57-dd1e645835d6
- CryptoGlobal_Weekly_20

# Graph Visualization (Interactive HTML)

In [24]:
import webbrowser

await cognee.visualize_graph("graph_Neo4j.html", dataset="docspace", full=True)
webbrowser.open("graph_Neo4j.html")


2026-11-13T22:44:52.121006 [info     ] Retrieved 416 nodes and 1013 edges in 0.74 seconds [Neo4jAdapter]

2026-11-13T22:44:52.288519 [info     ] Collected 1 session operation events for the Memory tab. [visualization.session_events]

2026-11-13T22:45:12.545412 [info     ] fetch_node_embeddings: resolved 348/416 node embeddings across 5 collection(s) [embedding_join]

2026-11-13T22:45:15.338032 [info     ] Graph visualization saved as graph_Neo4j.html [cognee.shared.logging_utils]

2026-11-13T22:45:15.344041 [info     ] The HTML file has been stored at path: graph_Neo4j.html [cognee.shared.logging_utils]


True

# Queryin the graph for cryptocurrency information (Search type: NATURAL_LANGUAGE)

In [16]:
CYPHER_RULES = (
    "Cypher rules: never use query parameters ($name, $substring, ...). "
    "Inline every literal value directly in the query."
)

user_question = "Which 10 entities are mentioned in the highest number of distinct document chunks? Return their names and chunk counts, ranked from highest to lowest."

results = await cognee.recall(
    query_text=(
        f"{user_question}\n{CYPHER_RULES}"
    ),
    query_type=cognee.SearchType.NATURAL_LANGUAGE,
)

for result in results:
    print(result)


2026-11-13T22:27:58.999001 [info     ] Starting attempt 1/3 for query generation [NaturalLanguageRetriever]

2026-11-13T22:28:20.048782 [info     ] Executing generated Cypher query (attempt 1): MATCH (dc:DocumentChunk)-[]-(e:Entity)
WITH e, COUNT(DISTINCT dc) AS chunk_count
RETURN e.name AS en... [NaturalLanguageRetriever]

2026-11-13T22:28:20.166521 [info     ] Successfully executed query (attempt 1): returned 10 result(s) [NaturalLanguageRetriever]

2026-11-13T22:28:20.218479 [info     ] recall: 10 results across sources=['graph'] (session=-) [recall]


kind='natural_language' search_type='NATURAL_LANGUAGE' text='{"entity_name": "stablecoins", "chunk_count": 7}' system_prompt=None score=None dataset_id='7a0653e7-4039-5cf0-9c42-5b6d1221541f' dataset_name='docspace' metadata={} raw={'entity_name': 'stablecoins', 'chunk_count': 7} structured=None source='graph'
kind='natural_language' search_type='NATURAL_LANGUAGE' text='{"entity_name": "regulatory arbitrage", "chunk_count": 4}' system_prompt=None score=None dataset_id='7a0653e7-4039-5cf0-9c42-5b6d1221541f' dataset_name='docspace' metadata={} raw={'entity_name': 'regulatory arbitrage', 'chunk_count': 4} structured=None source='graph'
kind='natural_language' search_type='NATURAL_LANGUAGE' text='{"entity_name": "bitcoin", "chunk_count": 4}' system_prompt=None score=None dataset_id='7a0653e7-4039-5cf0-9c42-5b6d1221541f' dataset_name='docspace' metadata={} raw={'entity_name': 'bitcoin', 'chunk_count': 4} structured=None source='graph'
kind='natural_language' search_type='NATURAL_LANGUAGE' te

In [ ]:
# Retrieve the schema of the graph: nodes and relationships
nodes = await graph.query("""
    MATCH (n:__Node__)
    RETURN DISTINCT n.type AS type
""")

edges = await graph.query("""
    MATCH (source:__Node__)-[r]->(target:__Node__)
    RETURN DISTINCT
        source.type AS source,
        type(r) AS relation,
        target.type AS target
""")

graph_schema = (
    "Node types: "
    + ", ".join(node["type"] for node in nodes)
    + "\nRelationships:\n"
    + "\n".join(
        f"{edge['source']} -[{edge['relation']}]-> {edge['target']}"
        for edge in edges
    )
)

# User question
user_question = (
    "Which documents discuss both stablecoins and regulatory arbitrage? "
    "Return the document titles without duplicates."
)

# Recall relevant information from the graph based on the user question and schema
results = await cognee.recall(
    query_text=(
        f"Question:\n{user_question}"
    ),
    query_type=cognee.SearchType.NATURAL_LANGUAGE,
    system_prompt=(
        f"You are a Cypher query assistant. Answer the question based on the provided graph schema and rules.\n"
        f"Graph schema:\n{graph_schema}"
        f"\nRules:\n{CYPHER_RULES}"
    )
)

for result in results:
    print(result)


2026-11-13T23:24:27.348480 [info     ] Starting attempt 1/3 for query generation [NaturalLanguageRetriever]

2026-11-13T23:24:49.463481 [info     ] Executing generated Cypher query (attempt 1): MATCH (d:TextDocument)--(e1:Entity {name: $topic1}),
      (d)--(e2:Entity {name: $topic2})
RETURN D... [NaturalLanguageRetriever]


╭─────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────╮
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\graph\neo4j_driver\adapter.py:273 in  │
│ query                                                                                                           │
│                                                                                                                 │
│    270 │   │   │                                                                                                │
│    271 │   │   │   try:                                                                                         │
│    272 │   │   │   │   async with self.get_session() as session:                                                │
│ ❱  273 │   │   │   │   │   result = await session.run(query, parameters=params)                                 │
│    274 │   │   │   │   │   data = await result.data()                                                           │
│    275 │   │   │   │   │   otel_span.set_attribute(COGNEE_DB_ROW_COUNT, len(data))                              │
│    276 │   │   │   │   │   return data                                                                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\neo4j\_async\work\session.py:332 in run                               │
│                                                                                                                 │
│   329 │   │   )                                                                                                 │
│   330 │   │   bookmarks = await self._get_bookmarks()                                                           │
│   331 │   │   parameters = dict(parameters or {}, **kwargs)                                                     │
│ ❱ 332 │   │   await self._auto_result._run(                                                                     │
│   333 │   │   │   query,                                                                                        │
│   334 │   │   │   parameters,                                                                                   │
│   335 │   │   │   self._config.database,                                                                        │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\neo4j\_async\work\result.py:236 in _run                               │
│                                                                                                                 │
│   233 │   │   )                                                                                                 │
│   234 │   │   self._pull()                                                                                      │
│   235 │   │   await self._connection.send_all()                                                                 │
│ ❱ 236 │   │   await self._attach()                                                                              │
│   237 │                                                                                                         │
│   238 │   def _pull(self):                                                                                      │
│   239 │   │   def on_records(records):                                                                          │
│                                                       


2026-11-13T23:24:49.606479 [error    ] Neo4j query error              [Neo4jAdapter] exception_message='{code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): topic1, topic2}' exception_type=ClientError traceback=True



╭─────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────╮
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\modules\retrieval\natural_language_retriever.py:145 in         │
│ _execute_cypher_query                                                                                           │
│                                                                                                                 │
│   142 │   │   │   │   │   if len(cypher_query) > 100                                                            │
│   143 │   │   │   │   │   else cypher_query                                                                     │
│   144 │   │   │   │   )                                                                                         │
│ ❱ 145 │   │   │   │   context = await graph_engine.query(cypher_query)                                          │
│   146 │   │   │   │                                                                                             │
│   147 │   │   │   │   if isinstance(context, list):                                                             │
│   148 │   │   │   │   │   # Internal nodes (e.g. per-user preference state) must never                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\utils\closing_lru_cache.py:443 in     │
│ await_with_lease                                                                                                │
│                                                                                                                 │
│    440 │   │   │   │   │   # The closure keeps ``self`` alive until the awaitable                               │
│    441 │   │   │   │   │   # completes, which matters for one-liners like                                       │
│    442 │   │   │   │   │   # ``await get_vector_engine_async().search(...)``.                                   │
│ ❱  443 │   │   │   │   │   return await result                                                                  │
│    444 │   │   │   │                                                                                            │
│    445 │   │   │   │   return await_with_lease()                                                                │
│    446                                                                                                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\graph\neo4j_driver\deadlock_retry.py: │
│ 43 in wrapper                                                                                                   │
│                                                                                                                 │
│   40 │   │   │   while attempt <= max_retries:                                                                  │
│   41 │   │   │   │   try:                                                                                       │
│   42 │   │   │   │   │   attempt += 1                                                                           │
│ ❱ 43 │   │   │   │   │   return await func(self, *args, **kwargs)                                               │
│   44 │   │   │   │   except DatabaseUnavailable:                                                                │
│   45 │   │   │   │   │   if attempt > max_retries:    


2026-11-13T23:24:50.125892 [error    ] Error executing query          [NaturalLanguageRetriever] exception_message='{code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): topic1, topic2}' exception_type=ClientError traceback=True


2026-11-13T23:24:50.696343 [info     ] Starting attempt 2/3 for query generation [NaturalLanguageRetriever]

2026-11-13T23:25:08.104894 [info     ] Executing generated Cypher query (attempt 2): MATCH (d:TextDocument)--(e1:Entity {name: $topic1}), (d)--(e2:Entity {name: $topic2})
RETURN DISTINC... [NaturalLanguageRetriever]


╭─────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────╮
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\graph\neo4j_driver\adapter.py:273 in  │
│ query                                                                                                           │
│                                                                                                                 │
│    270 │   │   │                                                                                                │
│    271 │   │   │   try:                                                                                         │
│    272 │   │   │   │   async with self.get_session() as session:                                                │
│ ❱  273 │   │   │   │   │   result = await session.run(query, parameters=params)                                 │
│    274 │   │   │   │   │   data = await result.data()                                                           │
│    275 │   │   │   │   │   otel_span.set_attribute(COGNEE_DB_ROW_COUNT, len(data))                              │
│    276 │   │   │   │   │   return data                                                                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\neo4j\_async\work\session.py:332 in run                               │
│                                                                                                                 │
│   329 │   │   )                                                                                                 │
│   330 │   │   bookmarks = await self._get_bookmarks()                                                           │
│   331 │   │   parameters = dict(parameters or {}, **kwargs)                                                     │
│ ❱ 332 │   │   await self._auto_result._run(                                                                     │
│   333 │   │   │   query,                                                                                        │
│   334 │   │   │   parameters,                                                                                   │
│   335 │   │   │   self._config.database,                                                                        │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\neo4j\_async\work\result.py:236 in _run                               │
│                                                                                                                 │
│   233 │   │   )                                                                                                 │
│   234 │   │   self._pull()                                                                                      │
│   235 │   │   await self._connection.send_all()                                                                 │
│ ❱ 236 │   │   await self._attach()                                                                              │
│   237 │                                                                                                         │
│   238 │   def _pull(self):                                                                                      │
│   239 │   │   def on_records(records):                                                                          │
│                                                       

... (3 outputs left out)

╭─────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────╮
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\graph\neo4j_driver\adapter.py:273 in  │
│ query                                                                                                           │
│                                                                                                                 │
│    270 │   │   │                                                                                                │
│    271 │   │   │   try:                                                                                         │
│    272 │   │   │   │   async with self.get_session() as session:                                                │
│ ❱  273 │   │   │   │   │   result = await session.run(query, parameters=params)                                 │
│    274 │   │   │   │   │   data = await result.data()                                                           │
│    275 │   │   │   │   │   otel_span.set_attribute(COGNEE_DB_ROW_COUNT, len(data))                              │
│    276 │   │   │   │   │   return data                                                                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\neo4j\_async\work\session.py:332 in run                               │
│                                                                                                                 │
│   329 │   │   )                                                                                                 │
│   330 │   │   bookmarks = await self._get_bookmarks()                                                           │
│   331 │   │   parameters = dict(parameters or {}, **kwargs)                                                     │
│ ❱ 332 │   │   await self._auto_result._run(                                                                     │
│   333 │   │   │   query,                                                                                        │
│   334 │   │   │   parameters,                                                                                   │
│   335 │   │   │   self._config.database,                                                                        │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\neo4j\_async\work\result.py:236 in _run                               │
│                                                                                                                 │
│   233 │   │   )                                                                                                 │
│   234 │   │   self._pull()                                                                                      │
│   235 │   │   await self._connection.send_all()                                                                 │
│ ❱ 236 │   │   await self._attach()                                                                              │
│   237 │                                                                                                         │
│   238 │   def _pull(self):                                                                                      │
│   239 │   │   def on_records(records):                                                                          │
│                                                       


2026-11-13T23:25:33.864417 [error    ] Neo4j query error              [Neo4jAdapter] exception_message='{code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): topic1, topic2}' exception_type=ClientError traceback=True



╭─────────────────────────────────────── Traceback (most recent call last) ───────────────────────────────────────╮
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\modules\retrieval\natural_language_retriever.py:145 in         │
│ _execute_cypher_query                                                                                           │
│                                                                                                                 │
│   142 │   │   │   │   │   if len(cypher_query) > 100                                                            │
│   143 │   │   │   │   │   else cypher_query                                                                     │
│   144 │   │   │   │   )                                                                                         │
│ ❱ 145 │   │   │   │   context = await graph_engine.query(cypher_query)                                          │
│   146 │   │   │   │                                                                                             │
│   147 │   │   │   │   if isinstance(context, list):                                                             │
│   148 │   │   │   │   │   # Internal nodes (e.g. per-user preference state) must never                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\utils\closing_lru_cache.py:443 in     │
│ await_with_lease                                                                                                │
│                                                                                                                 │
│    440 │   │   │   │   │   # The closure keeps ``self`` alive until the awaitable                               │
│    441 │   │   │   │   │   # completes, which matters for one-liners like                                       │
│    442 │   │   │   │   │   # ``await get_vector_engine_async().search(...)``.                                   │
│ ❱  443 │   │   │   │   │   return await result                                                                  │
│    444 │   │   │   │                                                                                            │
│    445 │   │   │   │   return await_with_lease()                                                                │
│    446                                                                                                          │
│                                                                                                                 │
│ d:\Documents\Vie Professionnelle\GitHub                                                                         │
│ Projects\docspace\.venv\Lib\site-packages\cognee\infrastructure\databases\graph\neo4j_driver\deadlock_retry.py: │
│ 43 in wrapper                                                                                                   │
│                                                                                                                 │
│   40 │   │   │   while attempt <= max_retries:                                                                  │
│   41 │   │   │   │   try:                                                                                       │
│   42 │   │   │   │   │   attempt += 1                                                                           │
│ ❱ 43 │   │   │   │   │   return await func(self, *args, **kwargs)                                               │
│   44 │   │   │   │   except DatabaseUnavailable:                                                                │
│   45 │   │   │   │   │   if attempt > max_retries:    


2026-11-13T23:25:34.411856 [error    ] Error executing query          [NaturalLanguageRetriever] exception_message='{code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): topic1, topic2}' exception_type=ClientError traceback=True


2026-11-13T23:25:35.078770 [warning  ] Failed to get results after 3 attempts for query: 'Question:
Which documents discuss both stablecoins...' [NaturalLanguageRetriever]

2026-11-13T23:25:35.160752 [info     ] recall: 0 results across sources=['graph'] (session=-) [recall]


In [46]:
import os
import re
import requests

# 1. Retrieve the graph schema
nodes = await graph.query("""
    MATCH (n:__Node__)
    RETURN DISTINCT n.type AS type
""")

edges = await graph.query("""
    MATCH (source:__Node__)-[r]->(target:__Node__)
    RETURN DISTINCT
        source.type AS source,
        type(r) AS relation,
        target.type AS target
""")

properties = await graph.query("""
    MATCH (n:__Node__)
    UNWIND keys(n) AS property
    RETURN DISTINCT n.type AS type, property
    ORDER BY type, property
""")

graph_schema = (
    "Node types:\n"
    + "\n".join(f"- {node['type']}" for node in nodes)
    + "\n\nRelationships:\n"
    + "\n".join(
        f"- {edge['source']} -[{edge['relation']}]-> {edge['target']}"
        for edge in edges
    )
    + "\n\nProperties:\n"
    + "\n".join(
        f"- {prop['type']}: {prop['property']}"
        for prop in properties
    )
)

# 2. User question
user_question = (
    "Which documents discuss both stablecoins and regulatory arbitrage? "
    "Return the document titles without duplicates."
)

# 3. Generate Cypher directly with OpenRouter
system_prompt = f"""
You are an expert Neo4j Cypher query generator.

Your task is to translate a natural language question into
a valid, read-only Neo4j Cypher query.

GRAPH SCHEMA:
{graph_schema}

RULES:
- Return ONLY the Cypher query.
- Do not include explanations or markdown.
- Use only node types and relationships present in the schema.
- Node types are stored in the `type` property.
- All nodes have the `__Node__` label.
- Do not assume that a node type is a Neo4j label.
- Do not invent node properties.
- Use DISTINCT when the user requests unique results.
- Prefer case-insensitive matching for text searches.
- Never generate CREATE, MERGE, DELETE, SET, REMOVE, DROP, or CALL.
- Return at most 100 rows.
"""

response = requests.post(
    "https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": f"Bearer {os.environ['LLM_API_KEY']}",
        "Content-Type": "application/json",
    },
    json={
        "model": "qwen/qwen3-coder-30b-a3b-instruct",
        "temperature": 0,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    },
    timeout=60,
)

response.raise_for_status()

cypher = response.json()["choices"][0]["message"]["content"].strip()

# Remove possible markdown fences
cypher = re.sub(r"^```(?:cypher)?\s*", "", cypher, flags=re.I)
cypher = re.sub(r"\s*```$", "", cypher).strip()

print("GENERATED CYPHER:\n")
print(cypher)

GENERATED CYPHER:

MATCH (d:TextDocument)<-[:is_part_of]-(dc:DocumentChunk)
WHERE (dc.text CONTAINS "stablecoin" OR dc.text CONTAINS "stablecoins")
  AND (dc.text CONTAINS "regulatory arbitrage" OR dc.text CONTAINS "arbitrage")
RETURN DISTINCT d.name AS document_title
LIMIT 100
